In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

# Load raw data (city_day.csv जैसे)
df = pd.read_csv('city_day.csv')

# Missing values handling
df.fillna(method='ffill', inplace=True)

# Pollutant features
features = ['PM2.5', 'PM10', 'NO2', 'CO', 'O3']

for col in features:
    df[col] = df[col].fillna(df[col].median())

# Convert Date if available
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])

# Normalize features
scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])

# AQI Category creation
def categorize_aqi(aqi):
    if aqi <= 50:
        return 'Good'
    elif aqi <= 100:
        return 'Moderate'
    elif aqi <= 200:
        return 'Poor'
    else:
        return 'Hazardous'

df['AQI_Category'] = df['AQI'].apply(categorize_aqi)

# Encode AQI Category labels
le = LabelEncoder()
df['AQI_Label'] = le.fit_transform(df['AQI_Category'])

# Prepare train/test split
X = df[features]
y = df['AQI_Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Decision Tree model
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Save model, label encoder, and scaler
joblib.dump(model, 'decision_tree_aqi_model.pkl')
joblib.dump(le, 'label_encoder.pkl')
joblib.dump(scaler, 'scaler.pkl')

# Save preprocessed dataset optionally
df.to_csv('preprocessed_airquality_data.csv', index=False)


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_12392\3423916430.py:12: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)


Accuracy: 0.7305288055414766
              precision    recall  f1-score   support

        Good       0.64      0.58      0.61       344
   Hazardous       0.85      0.76      0.80      1495
    Moderate       0.68      0.79      0.73      2080
        Poor       0.73      0.67      0.70      2000

    accuracy                           0.73      5919
   macro avg       0.72      0.70      0.71      5919
weighted avg       0.74      0.73      0.73      5919

